In [1]:
import ast
import os
import re
import json
import subprocess
import shutil
from dataclasses import dataclass, asdict
from pathlib import Path
from urllib.parse import urlparse

In [2]:
def clone_repo(url: str, dest_root: str='./temp/repos/')->Path:
    """"Clone a github repo and return the local path.
    re-Clones the cleany if the destination already exist."""

    parsed=urlparse(url)
    repo_name=Path(parsed.path).stem #owner/name.git -> name
    dest=Path(dest_root)/repo_name

    if dest.exists():
        shutil.rmtree(dest)
    dest.parent.mkdir(parents=True, exist_ok=True)

    result=subprocess.run(
        ['git', 'clone', '--depth', '1', url, str(dest)],
        capture_output=True, text=True,
    )

    if result.returncode!=0:
        raise RuntimeError(f"git clone failed: {result.stderr.strip()}")
    
    sha_result = subprocess.run(['git', '-C', str(dest), 'rev-parse', 'HEAD'],
                                 capture_output=True, text=True)
    commit_sha = sha_result.stdout.strip()[:12]  # short SHA is enough

    
    return dest, repo_name, commit_sha

In [3]:
repo_url = "https://github.com/shreeragkh/Hybrid-Search-RAG"

repo_path, repo_name, commit_sha = clone_repo(repo_url)
print(f"cloned {repo_name} ({commit_sha}) -> {repo_path}")
print(f"Files on disk: {sum(1 for _ in repo_path.rglob('*') if _.is_file())}")

cloned Hybrid-Search-RAG (48cbc7a6fdac) -> temp/repos/Hybrid-Search-RAG
Files on disk: 71


In [4]:
CODE_ONLY_MAP = {
    ".py": "python", ".js": "javascript", ".jsx": "javascript",
    ".ts": "typescript", ".tsx": "typescript", ".java": "java",
    ".go": "go", ".rb": "ruby", ".rs": "rust", ".c": "c", ".h": "c",
    ".cpp": "cpp", ".hpp": "cpp", ".cs": "csharp", ".php": "php",
}

EXCLUDE_DIRS = {".git", "node_modules", "venv", ".venv", "__pycache__",
                "dist", "build", ".next", "target", "vendor", ".idea", ".mypy_cache"}


EXCLUDE_FILENAMES = {"package-lock.json", "yarn.lock", "poetry.lock"}
EXCLUDE_PATTERNS = re.compile(r"\.min\.(js|css)$|\.d\.ts$|_pb2\.py$")
DOC_EXTENSIONS = {".md": "markdown", ".rst": "restructuredtext", ".txt": "text"}
PRIORITY_DOC_FILENAMES = {"readme.md", "readme.rst", "readme.txt", "readme"}

def discover_files(repo_dir: Path, max_file_kb: int = 500):
    files = []
    for root, dirs, filenames in os.walk(repo_dir):
        dirs[:] = [d for d in dirs if d not in EXCLUDE_DIRS and not d.startswith(".")]
        for fn in filenames:
            if fn in EXCLUDE_FILENAMES or EXCLUDE_PATTERNS.search(fn):
                continue
            ext = Path(fn).suffix.lower()
            is_code = ext in CODE_ONLY_MAP
            is_doc = ext in DOC_EXTENSIONS or fn.lower() in PRIORITY_DOC_FILENAMES
            if not (is_code or is_doc):
                continue
            full = Path(root) / fn
            try:
                if full.stat().st_size > max_file_kb * 1024:
                    continue
            except OSError:
                continue
            files.append(full)
    return files


def chunk_markdown_file(path: Path, repo_name: str):
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()
    header_pattern = re.compile(r"^#{1,3}\s+(.+)")
    starts = [i for i, line in enumerate(lines) if header_pattern.match(line)]
    if not starts:
        return chunk_generic_lines(path, repo_name, text, "markdown", window=80, overlap=10)

    chunks = []
    for idx, start in enumerate(starts):
        end = starts[idx + 1] - 1 if idx + 1 < len(starts) else len(lines) - 1
        while end > start and not lines[end].strip():
            end -= 1
        src = "\n".join(lines[start:end + 1])
        name = header_pattern.match(lines[start]).group(1).strip()
        chunks.append(Chunk(repo_name, str(path), "markdown", "doc_section",
                             name, start + 1, end + 1, src, len(src)))
    return chunks

In [5]:

discovered = discover_files(repo_path)
print(f"Discovered {len(discovered)} chunkable files")
from collections import Counter
print(Counter(f.suffix for f in discovered).most_common())

Discovered 23 chunkable files
[('.py', 21), ('.md', 1), ('.txt', 1)]


#### Chunking

In [6]:
@dataclass
class Chunk:
    repo: str
    file_path: str
    language: str
    symbol_type: str   # "function" | "class" | "method" | "block" | "file"
    symbol_name: str
    start_line: int
    end_line: int
    content: str
    char_count: int
    chunk_id: str = ""

    def __post_init__(self):
        if not self.chunk_id:
            raw = f"{self.repo}:{self.file_path}:{self.symbol_name}:{self.start_line}-{self.end_line}"
            self.chunk_id = hashlib.sha1(raw.encode()).hexdigest()[:16]


GENERIC_FUNC_PATTERNS = {
    "javascript": re.compile(r"^\s*(export\s+)?(async\s+)?function\s+(\w+)|^\s*(export\s+)?class\s+(\w+)|^\s*const\s+(\w+)\s*=\s*(async\s*)?\("),
    "typescript": re.compile(r"^\s*(export\s+)?(async\s+)?function\s+(\w+)|^\s*(export\s+)?class\s+(\w+)|^\s*const\s+(\w+)\s*=\s*(async\s*)?\("),
    "java": re.compile(r"^\s*(public|private|protected)?\s*(static\s+)?[\w<>\[\]]+\s+(\w+)\s*\("),
    "go": re.compile(r"^\s*func\s+(\(\w+\s+\*?\w+\)\s+)?(\w+)\s*\("),
    "ruby": re.compile(r"^\s*def\s+(\w+)|^\s*class\s+(\w+)"),
    "rust": re.compile(r"^\s*(pub\s+)?fn\s+(\w+)|^\s*(pub\s+)?struct\s+(\w+)"),
    "c": re.compile(r"^\s*[\w\*]+\s+(\w+)\s*\([^;]*\)\s*\{"),
    "cpp": re.compile(r"^\s*[\w\*:<>]+\s+(\w+)\s*\([^;]*\)\s*\{"),
    "csharp": re.compile(r"^\s*(public|private|protected)?\s*(static\s+)?[\w<>\[\]]+\s+(\w+)\s*\("),
    "php": re.compile(r"^\s*function\s+(\w+)|^\s*class\s+(\w+)"),
}

MAX_CHUNK_CHARS = 3000

def split_oversized(chunk: Chunk, max_chars: int = MAX_CHUNK_CHARS):
    if chunk.char_count <= max_chars:
        return [chunk]
    lines = chunk.content.splitlines()
    out, buf, buf_start = [], [], chunk.start_line
    cur_len = 0
    for i, line in enumerate(lines):
        buf.append(line)
        cur_len += len(line) + 1
        if cur_len >= max_chars:
            src = "\n".join(buf)
            out.append(Chunk(chunk.repo, chunk.file_path, chunk.language, chunk.symbol_type,
                              f"{chunk.symbol_name}_part{len(out)+1}", buf_start,
                              buf_start + len(buf) - 1, src, len(src)))
            buf, buf_start, cur_len = [], chunk.start_line + i + 1, 0
    if buf:
        src = "\n".join(buf)
        out.append(Chunk(chunk.repo, chunk.file_path, chunk.language, chunk.symbol_type,
                          f"{chunk.symbol_name}_part{len(out)+1}", buf_start,
                          buf_start + len(buf) - 1, src, len(src)))
    return out

def chunk_generic_lines(path: Path, repo_name: str, text: str, language: str, window: int = 60, overlap: int = 10):
    """Sliding-window line chunks with overlap. Fallback for languages/files
    without symbol-level parsing, or when a parse attempt fails."""
    lines = text.splitlines()
    chunks = []
    i, n = 0, len(lines)
    if n == 0:
        return chunks
    while i < n:
        end = min(i + window, n)
        src = "\n".join(lines[i:end])
        if src.strip():
            chunks.append(Chunk(repo_name, str(path), language, "block",
                                 f"lines_{i+1}-{end}", i + 1, end, src, len(src)))
        if end == n:
            break
        i += window - overlap
    return chunks


def chunk_python_file(path: Path, repo_name: str):
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()
    try:
        tree = ast.parse(text)
    except SyntaxError:
        return chunk_generic_lines(path, repo_name, text, "python")

    chunks, covered = [], set()

    def node_source(node):
        start = node.lineno
        end = getattr(node, "end_lineno", start)
        covered.update(range(start, end + 1))
        return start, end, "\n".join(lines[start - 1:end])

    for node in ast.iter_child_nodes(tree):
        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):
            start, end, src = node_source(node)
            chunks.append(Chunk(repo_name, str(path), "python", "function",
                                node.name, start, end, src, len(src)))
        elif isinstance(node, ast.ClassDef):
            start, end, src = node_source(node)
            chunks.append(Chunk(repo_name, str(path), "python", "class",
                                node.name, start, end, src, len(src)))
            for sub in node.body:
                if isinstance(sub, (ast.FunctionDef, ast.AsyncFunctionDef)):
                    s2, e2, src2 = node_source(sub)
                    chunks.append(Chunk(repo_name, str(path), "python", "method",
                                        f"{node.name}.{sub.name}", s2, e2, src2, len(src2)))
        elif isinstance(node, (ast.Assign, ast.AnnAssign)):
            start, end, src = node_source(node)
            if isinstance(node, ast.Assign) and node.targets:
                name = getattr(node.targets[0], "id", "constant")
            elif isinstance(node, ast.AnnAssign) and getattr(node.target, "id", None):
                name = node.target.id
            else:
                name = "constant"
            chunks.append(Chunk(repo_name, str(path), "python", "constant",
                                name, start, end, src, len(src)))

    leftover = [i + 1 for i in range(len(lines)) if (i + 1) not in covered]
    if leftover:
        start, end = min(leftover), max(leftover)
        src = "\n".join(lines[start - 1:end])
        if src.strip():
            chunks.append(Chunk(repo_name, str(path), "python", "block",
                                 "module_level", start, end, src, len(src)))
    return chunks


def chunk_generic_symbols(path: Path, repo_name: str, language: str):
    text = path.read_text(encoding="utf-8", errors="ignore")
    lines = text.splitlines()
    pattern = GENERIC_FUNC_PATTERNS.get(language)
    if pattern is None:
        return chunk_generic_lines(path, repo_name, text, language)

    starts = [i for i, line in enumerate(lines) if pattern.search(line)]
    if not starts:
        return chunk_generic_lines(path, repo_name, text, language)

    chunks = []
    for idx, start in enumerate(starts):
        end = starts[idx + 1] - 1 if idx + 1 < len(starts) else len(lines) - 1
        while end > start and not lines[end].strip():
            end -= 1
        src = "\n".join(lines[start:end + 1])
        m = pattern.search(lines[start])
        name = next((g for g in m.groups() if g and re.match(r"^\w+$", g)), "anonymous")
        chunks.append(Chunk(repo_name, str(path), language, "function",
                             name, start + 1, end + 1, src, len(src)))
    return chunks


def chunk_file(path: Path, repo_name: str):
    ext = path.suffix.lower()
    language = CODE_ONLY_MAP.get(ext, "text")
    if language == "python":
        return chunk_python_file(path, repo_name)
    if language in GENERIC_FUNC_PATTERNS:
        return chunk_generic_symbols(path, repo_name, language)
    if ext == ".md" or path.name.lower().startswith("readme"):
        return chunk_markdown_file(path, repo_name)
    if ext in DOC_EXTENSIONS:
        text = path.read_text(encoding="utf-8", errors="ignore")
        return chunk_generic_lines(path, repo_name, text, DOC_EXTENSIONS[ext])
    text = path.read_text(encoding="utf-8", errors="ignore")
    return chunk_generic_lines(path, repo_name, text, language)


def chunk_repo(repo_dir: Path, repo_name: str):
    all_chunks = []
    for f in discover_files(repo_dir):
        for c in chunk_file(f,repo_name):
            all_chunks.extend(split_oversized(c))
    return all_chunks


In [7]:
import hashlib

chunks = chunk_repo(repo_path, repo_name)

print(f"Total chunks: {len(chunks)}")
from collections import Counter
print("By symbol_type:", Counter(c.symbol_type for c in chunks))
print("By language:   ", Counter(c.language for c in chunks))
print(f"Avg chunk size: {sum(c.char_count for c in chunks) / len(chunks):.0f} chars")

print("\nSample chunks:")
for c in chunks[:20]:
    print(f"  [{c.language:10}] {c.symbol_type:8} {c.symbol_name:25} "
          f"{Path(c.file_path).name}:{c.start_line}-{c.end_line}")


Total chunks: 171
By symbol_type: Counter({'method': 37, 'block': 35, 'function': 30, 'constant': 28, 'class': 21, 'doc_section': 20})
By language:    Counter({'python': 150, 'markdown': 20, 'text': 1})
Avg chunk size: 843 chars

Sample chunks:
  [markdown  ] doc_section Hybrid Search RAG Application README.md:1-5
  [markdown  ] doc_section 🏗️ Architecture Overview  README.md:7-53
  [markdown  ] doc_section ✨ Key Features            README.md:55-71
  [markdown  ] doc_section 📁 Project Structure       README.md:73-120
  [markdown  ] doc_section 🛠️ Tech Stack & Models Used README.md:122-134
  [markdown  ] doc_section 🚀 Quick Start             README.md:136-136
  [markdown  ] doc_section 1. Prerequisites          README.md:138-143
  [markdown  ] doc_section 2. Environment Setup      README.md:145-151
  [markdown  ] doc_section Create virtual environment README.md:153-155
  [markdown  ] doc_section Install dependencies      README.md:157-159
  [markdown  ] doc_section 3. Environment Variab

In [8]:
OUT_PATH = f"./temp/repos/{repo_name}/chunks-{repo_name}.jsonl"

with open(OUT_PATH, "w", encoding="utf-8") as f:
    for c in chunks:
        f.write(json.dumps(asdict(c)) + "\n")

print(f"Wrote {len(chunks)} chunks to {OUT_PATH}")


Wrote 171 chunks to ./temp/repos/Hybrid-Search-RAG/chunks-Hybrid-Search-RAG.jsonl


#### Embedding and Db

In [9]:
import os
from dataclasses import asdict
from dotenv import load_dotenv
from astrapy import DataAPIClient
from astrapy.constants import VectorMetric
from sentence_transformers import SentenceTransformer
from astrapy.info import CollectionDefinition

load_dotenv()

# Load the embedding model locally (runs on your machine, free, no API calls)
model = SentenceTransformer("BAAI/bge-large-en-v1.5")

# Initialize the client
client = DataAPIClient()
db = client.get_database(
    api_endpoint=os.getenv("API_ENDPOINT"),
    token=os.getenv("API_TOKEN"),
)

definition = (
    CollectionDefinition.builder()
    .with_vector_dimension(1024)
    .with_vector_metric(VectorMetric.COSINE)
    .build()
)

# Drop the old collection if it exists — it was created with a `service` block,
# which is incompatible with bringing your own vectors. Must recreate clean.
if "repo_context" in db.list_collection_names():
    db.drop_collection("repo_context")

# Create collection WITHOUT a service block — no Astra-side embedding provider needed
collection = db.create_collection(
    "repo_context",
    definition=definition
)

# Compute embeddings locally
texts = [c.content for c in chunks]
vectors = model.encode(
    texts,
    normalize_embeddings=True,
    show_progress_bar=True,
    batch_size=32,
).tolist()

# Prepare documents with $vector (pre-computed), not $vectorize
documents = [{"_id": c.chunk_id, "$vector": vec, **asdict(c)} for vec, c in zip(vectors, chunks)]

# Insert chunks — no embedding provider call per-batch anymore, so no timeouts,
# can use a larger batch size and don't need retry logic for provider timeouts
batch_size = 50
all_inserted = []
for i in range(0, len(documents), batch_size):
    batch = documents[i:i + batch_size]
    result = collection.insert_many(batch, request_timeout_ms=30000)
    all_inserted.extend(result.inserted_ids)
    print(f"Batch {i // batch_size + 1}: inserted {len(result.inserted_ids)}")

print(f"\nSuccessfully inserted {len(all_inserted)} chunks into Astra DB!")
print(f"Collections in Astra DB: {db.list_collection_names()}")

/home/shreerag/Desktop/shreeragkh/Repo-context-copilot/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Batches: 100%|██████████| 6/6 [03:38<00:00, 36.41s/it]


Batch 1: inserted 50
Batch 2: inserted 50
Batch 3: inserted 50
Batch 4: inserted 21

Successfully inserted 171 chunks into Astra DB!
Collections in Astra DB: ['repo_context']


In [10]:
import json
import logging
from dataclasses import asdict, is_dataclass
from pathlib import Path
from typing import Any

import bm25s

logger = logging.getLogger(__name__)


class BM25Retriever:
    """
    Wraps bm25s.BM25 to support:
      - building an index from a list of chunk dicts or dataclasses (text + metadata)
      - persisting the index and metadata to disk, scoped per repo via index_dir
      - reloading without re-tokenizing the corpus
      - querying with scores, metadata, and chunk_id attached (for RRF fusion
        against vector search results keyed on the same chunk_id)
    """

    def __init__(self, index_dir: str | Path = "bm25_index"):
        self.index_dir = Path(index_dir)
        self.retriever: bm25s.BM25 | None = None
        self.corpus: list[str] = []
        self.metadata: list[dict[str, Any]] = []

    # ------------------------------------------------------------------
    # Helpers
    # ------------------------------------------------------------------
    @staticmethod
    def _to_dicts(chunks: list[Any]) -> list[dict[str, Any]]:
        """Accept a list of dicts or dataclass instances (e.g. Chunk) transparently."""
        return [asdict(c) if is_dataclass(c) else c for c in chunks]

    # ------------------------------------------------------------------
    # Build
    # ------------------------------------------------------------------
    def build(self, chunks: list[Any], text_key: str = "content") -> None:
        """
        Build the BM25 index from scratch.

        Args:
            chunks: list of dicts or dataclass instances, each containing at
                    least `text_key` (e.g. the Chunk dataclass's `content` field,
                    and ideally a `chunk_id` field for fusion with vector results).
                    All other keys are stored as metadata and returned
                    alongside results at query time.
            text_key: the dict key holding the chunk's raw text. Defaults to
                    "content" to match the project's Chunk dataclass.
        """
        chunk_dicts = self._to_dicts(chunks)
        if not chunk_dicts:
            raise ValueError("Cannot build BM25 index from an empty chunk list.")

        self.corpus = [c[text_key] for c in chunk_dicts]
        self.metadata = [{k: v for k, v in c.items() if k != text_key} for c in chunk_dicts]

        logger.info("Tokenizing %d chunks for BM25 indexing...", len(self.corpus))
        tokens = bm25s.tokenize(self.corpus, show_progress=False)

        self.retriever = bm25s.BM25()
        self.retriever.index(tokens, show_progress=False)
        logger.info("BM25 index built with %d documents.", len(self.corpus))

    # ------------------------------------------------------------------
    # Incremental-ish rebuild (bm25s has no true incremental add;
    # this re-tokenizes the full corpus with new chunks appended)
    # ------------------------------------------------------------------
    def add(self, chunks: list[Any], text_key: str = "content") -> None:
        """
        Append new chunks and rebuild the index. bm25s does not support
        true incremental indexing, so this re-indexes the full corpus.
        Fine for periodic batch updates (e.g. re-ingesting a repo); avoid
        calling this per-request.
        """
        chunk_dicts = self._to_dicts(chunks)
        new_texts = [c[text_key] for c in chunk_dicts]
        new_meta = [{k: v for k, v in c.items() if k != text_key} for c in chunk_dicts]

        self.corpus.extend(new_texts)
        self.metadata.extend(new_meta)

        logger.info("Rebuilding BM25 index with %d total documents...", len(self.corpus))
        tokens = bm25s.tokenize(self.corpus, show_progress=False)
        self.retriever = bm25s.BM25()
        self.retriever.index(tokens, show_progress=False)

    # ------------------------------------------------------------------
    # Persistence
    # ------------------------------------------------------------------
    def save(self) -> None:
        """Persist the BM25 index, corpus, and metadata to self.index_dir."""
        if self.retriever is None:
            raise RuntimeError("No index to save. Call build() first.")

        self.index_dir.mkdir(parents=True, exist_ok=True)

        # bm25s handles the index + corpus itself
        self.retriever.save(str(self.index_dir), corpus=self.corpus)

        # metadata isn't tracked by bm25s, so store it ourselves
        meta_path = self.index_dir / "metadata.json"
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(self.metadata, f)

        logger.info("Saved BM25 index and metadata to %s", self.index_dir)

    def load(self) -> None:
        """Load a previously saved index, corpus, and metadata from disk."""
        if not self.index_dir.exists():
            raise FileNotFoundError(f"No index found at {self.index_dir}")

        self.retriever = bm25s.BM25.load(str(self.index_dir), load_corpus=True)

        meta_path = self.index_dir / "metadata.json"
        if meta_path.exists():
            with open(meta_path, "r", encoding="utf-8") as f:
                self.metadata = json.load(f)
        else:
            logger.warning("No metadata.json found at %s; metadata will be empty.", self.index_dir)
            self.metadata = [{} for _ in range(len(self.retriever.corpus))]

        self.corpus = [doc["text"] if isinstance(doc, dict) else doc for doc in self.retriever.corpus]

        # Detach the corpus from the bm25s object so retrieve() always
        # returns plain indices rather than document text/dicts. This keeps
        # query() lookups simple and correct even with duplicate chunk text.
        self.retriever.corpus = None

        logger.info("Loaded BM25 index with %d documents from %s", len(self.corpus), self.index_dir)

    # ------------------------------------------------------------------
    # Query
    # ------------------------------------------------------------------
    def query(self, query_text: str, k: int = 10) -> list[dict[str, Any]]:
        """
        Retrieve top-k chunks for a query.

        Returns:
            list of dicts: {"text": ..., "score": ..., "chunk_id": ..., "metadata": {...}}
            sorted by descending BM25 score. `chunk_id` is lifted out of metadata
            (if present) to the top level so it can be joined directly against
            vector search results in an RRF fusion step.
        """
        if self.retriever is None:
            raise RuntimeError("Index not built or loaded. Call build() or load() first.")

        k = min(k, len(self.corpus))
        if k == 0:
            return []

        query_tokens = bm25s.tokenize(query_text, show_progress=False)
        doc_indices, scores = self.retriever.retrieve(query_tokens, k=k, show_progress=False)

        results = []
        for idx, score in zip(doc_indices[0], scores[0]):
            idx = int(idx)
            meta = self.metadata[idx] if idx < len(self.metadata) else {}
            results.append({
                "text": self.corpus[idx],
                "score": float(score),
                "chunk_id": meta.get("chunk_id"),
                "metadata": meta,
            })
        return results

    def __len__(self) -> int:
        return len(self.corpus)

In [11]:
bm25=BM25Retriever(index_dir=f"./bm25_index/{repo_name}")
bm25.build(chunks)
bm25.save()

#### Retreiever

In [12]:
from typing import List, Dict, Any


class VectorRetriever:
    """Handles query-based semantic retrieval from the Astra DB vector store."""

    def __init__(self, collection, model):
        """
        Initialize the retriever pipeline for Astra DB.

        Args:
            collection: an astrapy Collection with pre-computed $vector fields
                        (see ingestion pipeline — chunks were embedded locally
                        with sentence-transformers and inserted as $vector).
            model: the same SentenceTransformer instance used at ingestion time.
                   Must match exactly, or query/document vectors won't be comparable.
        """
        self.collection = collection
        self.model = model

    def query(self, query_text: str, k: int = 5) -> List[Dict[str, Any]]:
        """Helper to match the query interface of other retrievers (e.g. BM25Retriever)."""
        return self.retrieve(query_text, top_k=k)

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        """
        Retrieve relevant chunks for a query via vector similarity search.

        Args:
            query: query from the user
            top_k: number of top results to return
            score_threshold: minimum similarity score threshold (0-1, cosine)

        Returns:
            List of dicts: {"chunk_id", "text", "content", "metadata",
            "score", "rank"} — shaped to match BM25Retriever.query() output
            so both can be merged directly in RRF fusion.
        """
        try:
            query_vector = self.model.encode(
                [query], normalize_embeddings=True
            ).tolist()[0]

            results = self.collection.find(
                sort={"$vector": query_vector},
                limit=top_k,
                include_similarity=True,
            )

            retrieved_docs = []
            for i, doc in enumerate(results):
                similarity_score = doc.get("$similarity", 0.0)
                if similarity_score < score_threshold:
                    continue

                content = doc.get("content", "")
                metadata = {
                    k: v for k, v in doc.items()
                    if k not in ("_id", "$vector", "$similarity", "content")
                }

                retrieved_docs.append({
                    "id": doc.get("_id"),
                    "chunk_id": doc.get("chunk_id", doc.get("_id")),
                    "text": content,
                    "content": content,
                    "metadata": metadata,
                    "score": similarity_score,
                    "similarity_score": similarity_score,
                    "rank": i + 1,
                })

            print(f"Retrieved documents: {len(retrieved_docs)} documents (after filtering)")
            return retrieved_docs

        except Exception as e:
            print(f"Error during retrieval: {e}")
            return []


# Usage
vector_retrieval = VectorRetriever(collection, model)  # `model` = your SentenceTransformer instance

#### HybridSearch

In [13]:
from __future__ import annotations
import logging
import time
from concurrent.futures import ThreadPoolExecutor, TimeoutError as FutureTimeoutError
from typing import Any, Callable

logger = logging.getLogger(__name__)


class HybridSearchError(Exception):
    """Raised only when BOTH retrievers fail — total retrieval failure."""


def _reciprocal_rank_fusion(
    bm25_results: list[dict],
    vector_results: list[dict],
    bm25_weight: float,
    vector_weight: float,
    rrf_k: int = 60,
    id_key: str = "chunk_id",
) -> list[dict]:

    fused_docs = {}

    def get_id(doc):
        # chunk_id is exposed at the TOP LEVEL by both BM25Retriever and
        # VectorRetriever (see their query() implementations), so check
        # there first. Fall back to metadata, then raw text as a last resort
        # for any retriever that doesn't provide a stable chunk_id.
        if doc.get(id_key):
            return str(doc[id_key])
        meta = doc.get("metadata") or {}
        if id_key in meta and meta[id_key]:
            return str(meta[id_key])
        text = doc.get("text") or doc.get("content") or ""
        return text.strip()

    for rank, doc in enumerate(bm25_results):
        doc_id = get_id(doc)
        text = doc.get("text") or doc.get("content") or ""
        metadata = doc.get("metadata") or {}
        score = doc.get("score", 0.0)

        fused_docs[doc_id] = {
            "chunk_id": doc.get(id_key) or doc_id,
            "text": text,
            "metadata": metadata,
            "bm25_score": score,
            "vector_score": 0.0,
            "bm25_rrf": bm25_weight * (1.0 / (rrf_k + (rank + 1))),
            "vector_rrf": 0.0,
        }

    for rank, doc in enumerate(vector_results):
        doc_id = get_id(doc)
        text = doc.get("text") or doc.get("content") or ""
        metadata = doc.get("metadata") or {}
        score = doc.get("similarity_score") or doc.get("score") or 0.0

        if doc_id in fused_docs:
            fused_docs[doc_id]["vector_score"] = score
            fused_docs[doc_id]["vector_rrf"] = vector_weight * (1.0 / (rrf_k + (rank + 1)))
            if not fused_docs[doc_id]["metadata"] and metadata:
                fused_docs[doc_id]["metadata"] = metadata
            if not fused_docs[doc_id]["text"] and text:
                fused_docs[doc_id]["text"] = text
        else:
            fused_docs[doc_id] = {
                "chunk_id": doc.get(id_key) or doc_id,
                "text": text,
                "metadata": metadata,
                "bm25_score": 0.0,
                "vector_score": score,
                "bm25_rrf": 0.0,
                "vector_rrf": vector_weight * (1.0 / (rrf_k + (rank + 1))),
            }

    output = []
    for doc_id, info in fused_docs.items():
        fused_score = info["bm25_rrf"] + info["vector_rrf"]
        output.append({
            "chunk_id": info["chunk_id"],
            "text": info["text"],
            "metadata": info["metadata"],
            "fused_score": fused_score,
            "bm25_score": info["bm25_score"],
            "vector_score": info["vector_score"],
        })

    output.sort(key=lambda x: x["fused_score"], reverse=True)
    return output


class HybridSearch:
    """Handles query-based hybrid search: BM25 + vector retrieval fused via RRF."""

    def __init__(self, bm25_retriever, vector_retriever):
        self.bm25_retriever = bm25_retriever
        self.vector_retriever = vector_retriever

    def hybrid_retrieval(
        self,
        query_text: str,
        k: int = 10,
        fetch_k: int = 25,
        bm25_weight: float = 0.4,
        vector_weight: float = 0.6,
        rrf_k: int = 60,
        id_key: str = "chunk_id",
        metadata_filter: Callable[[dict], bool] | None = None,
        timeout_s: float = 15.0,
    ) -> list[dict[str, Any]]:
        """
        Runs BM25 and vector retrieval in parallel, fuses with RRF, and returns top-k.

        Args:
            query_text: user's query, e.g. "how does the auth middleware work?"
            k: number of results to return after fusion.
            fetch_k: candidates pulled from EACH retriever before fusion.
            bm25_weight / vector_weight: RRF weighting between the two signals.
            rrf_k: RRF damping constant (60 is the standard default).
            id_key: field used as the stable dedup key across both retrievers.
                    Defaults to "chunk_id", set on every Chunk at ingestion time
                    and returned at the top level by both retrievers.
            metadata_filter: optional predicate applied after fusion, e.g.
                    lambda m: m.get("language") == "python".
            timeout_s: max seconds to wait for EACH retriever before treating
                    it as failed and falling back to the other.

        Returns:
            List of {"chunk_id", "text", "metadata", "fused_score",
            "bm25_score", "vector_score"} sorted by fused_score descending.

        Raises:
            HybridSearchError if both retrievers fail.
        """
        start = time.monotonic()
        bm25_results, vector_results = self._run_retrievers_with_fallback(
            query_text, fetch_k, timeout_s
        )

        fused = _reciprocal_rank_fusion(
            bm25_results, vector_results, bm25_weight, vector_weight, rrf_k, id_key
        )

        if metadata_filter is not None:
            fused = [r for r in fused if metadata_filter(r.get("metadata", {}))]
        results = fused[:k]

        logger.info(
            "hybrid_search query=%r bm25_hits=%d vector_hits=%d fused=%d returned=%d latency_ms=%.0f",
            query_text, len(bm25_results), len(vector_results), len(fused), len(results),
            (time.monotonic() - start) * 1000,
        )
        return results

    def _run_retrievers_with_fallback(
        self, query_text: str, fetch_k: int, timeout_s: float
    ) -> tuple[list[dict], list[dict]]:
        """Run both retrievers concurrently; a failure/timeout in one degrades
        gracefully to results from the other instead of raising."""

        def safe_call(fn, name: str) -> list[dict]:
            try:
                return fn(query_text, k=fetch_k)
            except Exception:
                logger.exception("Retriever %s failed", name)
                return []

        with ThreadPoolExecutor(max_workers=2) as executor:
            bm25_future = executor.submit(safe_call, self.bm25_retriever.query, "bm25")
            vector_future = executor.submit(safe_call, self.vector_retriever.query, "vector")

            try:
                bm25_results = bm25_future.result(timeout=timeout_s)
            except FutureTimeoutError:
                logger.warning("BM25 retriever timed out after %.1fs", timeout_s)
                bm25_results = []

            try:
                vector_results = vector_future.result(timeout=timeout_s)
            except FutureTimeoutError:
                logger.warning("Vector retriever timed out after %.1fs", timeout_s)
                vector_results = []

        if not bm25_results and not vector_results:
            raise HybridSearchError(f"Both retrievers failed or timed out for query: {query_text!r}")
        return bm25_results, vector_results


# Usage — bm25_retriever from BM25Retriever, vector_retriever from VectorRetriever
hybrid_search = HybridSearch(bm25, vector_retrieval)
hybrid_search.hybrid_retrieval("how does the authentication middleware work?")

Retrieved documents: 25 documents (after filtering)


[{'chunk_id': 'b36827bfe36fe682',
  'text': "## 🔐 Firebase Auth Setup\n\nTo set up Google Sign-In for Admin authentication:\n\n1. Create a Firebase Project in [Firebase Console](https://console.firebase.google.com/).\n2. Enable **Google Sign-In** under **Authentication → Sign-in method**.\n3. Add `localhost` under **Authentication → Settings → Authorized domains**.\n4. Update `.env` with your project's `FIREBASE_API_KEY`, `FIREBASE_AUTH_DOMAIN`, and `FIREBASE_PROJECT_ID`.\n5. Set `ADMIN_EMAIL` in `.env` to your authorized Google email address.\n\n---",
  'metadata': {'repo': 'Hybrid-Search-RAG',
   'file_path': 'temp/repos/Hybrid-Search-RAG/README.md',
   'language': 'markdown',
   'symbol_type': 'doc_section',
   'symbol_name': '🔐 Firebase Auth Setup',
   'start_line': 225,
   'end_line': 235,
   'char_count': 501,
   'chunk_id': 'b36827bfe36fe682'},
  'fused_score': 0.015380906460945035,
  'bm25_score': 2.3985211849212646,
  'vector_score': 0.7955601},
 {'chunk_id': 'ca2e416d26e7da24

#### Reranker

In [14]:
from __future__ import annotations
 
import logging
import time
from typing import Any, Protocol
 
logger = logging.getLogger(__name__)
 
DEFAULT_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
# Stronger, slower alternative: "BAAI/bge-reranker-base" or "BAAI/bge-reranker-large"
 
 
class ScoringBackend(Protocol):
    """Minimal interface a reranking backend must satisfy."""
    def predict(self, pairs: list[tuple[str, str]]) -> list[float]: ...
 
 
class RerankerError(Exception):
    """Raised when reranking fails and no safe fallback is possible."""
 
 
class Reranker:
    def __init__(
        self,
        model_name: str = DEFAULT_MODEL,
        batch_size: int = 32,
        device: str | None = None,
        backend: ScoringBackend | None = None,
    ):
        """
        Args:
            model_name: HuggingFace cross-encoder model id. Ignored if
                        `backend` is supplied.
            batch_size: pairs per forward pass. Tune to your GPU/CPU memory.
            device: "cuda", "cpu", or None to let sentence-transformers pick.
            backend: inject a custom scoring backend (e.g. a Cohere Rerank
                     wrapper) instead of loading a local model. Must expose
                     .predict(list[(query, doc_text)]) -> list[float].
        """
        self.batch_size = batch_size
        self.model_name = model_name
 
        if backend is not None:
            self.backend = backend
        else:
            self.backend = self._load_local_model(model_name, device)
 
    @staticmethod
    def _load_local_model(model_name: str, device: str | None):
        try:
            from sentence_transformers import CrossEncoder
        except ImportError as e:
            raise ImportError(
                "sentence-transformers is required for local reranking. "
                "Install with: pip install sentence-transformers --break-system-packages"
            ) from e
 
        logger.info("Loading cross-encoder reranker model: %s", model_name)
        model = CrossEncoder(model_name, device=device)
        return model
 
    def rerank(
        self,
        query: str,
        candidates: list[dict[str, Any]],
        top_n: int = 5,
        min_score: float | None = None,
        text_key: str = "text",
        fallback_on_error: bool = True,
    ) -> list[dict[str, Any]]:
        """
        Score each candidate against the query and return the top_n,
        re-sorted by cross-encoder relevance score.
 
        Args:
            query: the user query.
            candidates: list of dicts (as returned by hybrid_search), each
                        containing at least `text_key`.
            top_n: number of results to return after reranking.
            min_score: optional threshold; candidates scoring below this
                       are dropped even if within top_n. Use this to avoid
                       feeding clearly-irrelevant context to the LLM when
                       retrieval had a bad day.
            text_key: dict key holding each candidate's text.
            fallback_on_error: if True and scoring fails, return the
                       original candidates truncated to top_n rather than
                       raising — keeps the pipeline degrading gracefully
                       instead of hard-failing generation.
 
        Returns:
            List of candidate dicts (original fields preserved) with an
            added "rerank_score" key, sorted descending, length <= top_n.
        """
        if not candidates:
            return []
 
        start = time.monotonic()
        pairs = [(query, c[text_key]) for c in candidates]
 
        try:
            scores = self._score_in_batches(pairs)
        except Exception:
            logger.exception("Reranking failed for query=%r (%d candidates)", query, len(candidates))
            if fallback_on_error:
                logger.warning("Falling back to pre-rerank order (no cross-encoder scores applied).")
                return [{**c, "rerank_score": c.get("fused_score", 0.0)} for c in candidates[:top_n]]
            raise RerankerError(f"Reranking failed for query: {query!r}")
 
        scored = [
            {**cand, "rerank_score": float(score)}
            for cand, score in zip(candidates, scores)
        ]
        scored.sort(key=lambda c: c["rerank_score"], reverse=True)
 
        if min_score is not None:
            scored = [c for c in scored if c["rerank_score"] >= min_score]
 
        results = scored[:top_n]
 
        logger.info(
            "rerank query=%r candidates=%d returned=%d top_score=%.4f latency_ms=%.0f",
            query, len(candidates), len(results),
            results[0]["rerank_score"] if results else float("nan"),
            (time.monotonic() - start) * 1000,
        )
        return results
 
    def _score_in_batches(self, pairs: list[tuple[str, str]]) -> list[float]:
        scores: list[float] = []
        for i in range(0, len(pairs), self.batch_size):
            batch = pairs[i : i + self.batch_size]
            batch_scores = self.backend.predict(batch)
            scores.extend(float(s) for s in batch_scores)
        return scores


In [15]:
re_ranker=Reranker()
re_ranker

Loading weights: 100%|██████████| 105/105 [00:00<00:00, 1622.39it/s]


#### RAG PipeLine

In [16]:
import re
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()

RELEVANCE_THRESHOLD = -8.5
ABSTENTION = "Cannot be determined from the provided repository context."


classifier_llm = ChatOpenAI(
    model="gpt-5-nano",
    temperature=0,
    max_tokens=128,
    reasoning_effort="minimal",
    api_key=os.getenv("OPENAI_API_KEY"),
)

llm = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0,
    max_tokens=256,
    reasoning_effort="minimal",
    api_key=os.getenv("OPENAI_API_KEY"),
)


def _empty_result(
    message,
    complexity="UNKNOWN",
    complexity_conf=0.0,
    complexity_reason="",
    fetch_k=0,
    return_context=False,
    context="",
):
    result = {
        "answer": message,
        "sources": [],
        "confidence": 0.0,
        "complexity": complexity,
        "complexity_confidence": complexity_conf,
        "complexity_reason": complexity_reason,
        "retrieval_k": fetch_k,
        "final_chunk_count": 0,
    }

    if return_context:
        result["context"] = context

    return result


def ragPipeline(
    query,
    hybrid_search=None,
    reranker=None,
    llm=None,
    top_k=None,
    top_n=None,
    min_score=0.2,
    return_context=False,
    use_adaptive=True,
):
    """
    Hybrid retrieval + reranking + adaptive cutoff + LLM generation.
    """

    # Resolve dependencies
    if hybrid_search is None:
        hybrid_search = globals().get("hybrid_search")

    if hybrid_search is None:
        raise ValueError("hybrid_search was not provided.")

    if reranker is None:
        reranker = globals().get("re_ranker")

    if llm is None:
        llm = globals().get("llm")

    if llm is None:
        raise ValueError("llm was not provided.")

    # 1. Classify query complexity
    complexity, complexity_conf, complexity_reason = classify_complexity(
        query,
        classifier_llm,
    )

    # 2. Select retrieval budget
    retrieval_budget = {
        "LOW": 10,
        "MEDIUM": 20,
        "HIGH": 30,
    }

    fetch_k = top_k or retrieval_budget.get(
        complexity,
        retrieval_budget["MEDIUM"],
    )

    # 3. Hybrid retrieval
    results = hybrid_search.hybrid_retrieval(
        query,
        k=fetch_k,
    )

    if not results:
        return _empty_result(
            ABSTENTION,
            complexity,
            complexity_conf,
            complexity_reason,
            fetch_k,
            return_context,
        )

    # 4. Filter by fused retrieval score
    bm25_weight = 0.4
    vector_weight = 0.6
    rrf_k = 60

    max_rrf_score = (
        bm25_weight + vector_weight
    ) / (rrf_k + 1)

    filtered_results = []

    for doc in results:
        fused_score = doc.get("fused_score", 0.0)
        normalized_score = fused_score / max_rrf_score

        if normalized_score >= min_score:
            filtered_results.append(doc)

    results = filtered_results

    if not results:
        return _empty_result(
            ABSTENTION,
            complexity,
            complexity_conf,
            complexity_reason,
            fetch_k,
            return_context,
        )

    # 5. Reranking
    if reranker is not None:
        rerank_top_n = top_n or fetch_k
        results = reranker.rerank(
            query,
            results,
            top_n=rerank_top_n,
        )

    # 6. Adaptive cutoff
    if use_adaptive and results:
        results = adaptive_cutoff(
            results,
            complexity=complexity,
        )

    if not results:
        return _empty_result(
            ABSTENTION,
            complexity,
            complexity_conf,
            complexity_reason,
            fetch_k,
            return_context,
        )

    # 7. Remove exact duplicate chunks
    unique_results = []
    seen_chunks = set()

    for doc in results:
        text = doc.get("text", "").strip()
        metadata = doc.get("metadata", {})
        file_path = metadata.get("file_path", "unknown")

        key = (file_path, text)

        if key not in seen_chunks:
            seen_chunks.add(key)
            unique_results.append(doc)

    results = unique_results

    # 8. Trim context to token budget
    prompt_overhead = count_tokens(query) + 80

    results = trim_to_token_budget(
        results,
        tpm_limit=MODEL_CONTEXT_WINDOW,
        reserved_output_tokens=512,
        prompt_overhead_tokens=prompt_overhead,
        safety_margin=200,
    )

    if not results:
        return _empty_result(
            ABSTENTION,
            complexity,
            complexity_conf,
            complexity_reason,
            fetch_k,
            return_context,
        )

    # 9. Build context
    context = "\n\n".join(
        doc.get("text", "")
        for doc in results
    ).strip()

    # 10. Build source metadata
    sources = []

    for doc in results:
        metadata = doc.get("metadata", {})

        sources.append({
            "source": metadata.get("file_path", "unknown"),
            "page": metadata.get("page", "unknown"),
            "score": doc.get("fused_score", 0.0) / max_rrf_score,
            "preview": doc.get("text", "")[:120] + "...",
        })

    confidence = max(
        doc.get("fused_score", 0.0) / max_rrf_score
        for doc in results
    )

    # 11. Relevance-based abstention
    best_rerank_score = max(
        (
            doc.get(
                "rerank_score",
                float("-inf"),
            )
            for doc in results
        ),
        default=float("-inf"),
    )

    if (
        not context
        or best_rerank_score < RELEVANCE_THRESHOLD
    ):
        answer = ABSTENTION

    else:
        # 12. Final generation
        prompt = f"""Answer the question using only the repository context below.

Answer directly in 1–3 concise sentences.
Use relevant file names, functions, classes, endpoints, and code from the context.
Do not use outside knowledge.

Repository context:
{context}

Question:
{query}

Answer:"""

        try:
            response = llm.invoke(prompt)
            content = response.content

            if isinstance(content, list):
                content = "".join(
                    part.get("text", "")
                    if isinstance(part, dict)
                    else str(part)
                    for part in content
                )

            answer = str(content or "").strip()

            # Remove hidden reasoning blocks if returned
            answer = re.sub(
                r"<think>.*?</think>",
                "",
                answer,
                flags=re.DOTALL,
            ).strip()

            if not answer:
                answer = ABSTENTION

        except Exception as exc:
            return _empty_result(
                f"Generation failed: {exc}",
                complexity,
                complexity_conf,
                complexity_reason,
                fetch_k,
                return_context,
                context,
            )

    # 13. Final output
    output = {
        "answer": answer,
        "sources": sources,
        "confidence": confidence,
        "complexity": complexity,
        "complexity_confidence": complexity_conf,
        "complexity_reason": complexity_reason,
        "retrieval_k": fetch_k,
        "final_chunk_count": len(results),
    }

    if return_context:
        output["context"] = context

    return output

In [17]:
COMPLEXITY_PROMPT = """
You are a query complexity classifier for a code repository.

Your task is to determine how much repository context is likely
required to answer the user's query.

Classify the query into exactly one of these levels:

LOW:
- Can probably be answered from one file, function, class, or
  small local section.
- Does not require significant cross-file reasoning.

MEDIUM:
- Requires understanding multiple related files, functions,
  or components.
- May require following a limited data or execution flow.

HIGH:
- Requires understanding multiple components or subsystems.
- Requires tracing a multi-step execution or data flow.
- Requires architectural or dependency reasoning.
- Requires understanding how several parts of the repository interact.

Consider:
1. Number of components involved
2. Number of files likely to be required
3. Whether the query requires tracing a flow
4. Whether cross-file reasoning is required
5. Whether architectural reasoning is required
6. Whether multiple steps need to be understood
7. Whether the query asks for comparison or impact analysis

Do not classify based only on query length.

Return ONLY valid JSON:

{{
    "complexity": "LOW | MEDIUM | HIGH",
    "confidence": 0.0,
    "reason": "Brief explanation"
}}

User query:
{query}
"""

#### Classify complexity

In [18]:
import re

def classify_complexity_heuristic(query: str) -> str | None:
    """Fast, free heuristic. Returns None if uncertain -> triggers LLM fallback."""
    q = query.lower().strip()
    word_count = len(q.split())

    strong_high_signals = [
        "trace", "end to end", "end-to-end",
        "architecture", "across", "interact",
    ]

    if any(signal in q for signal in strong_high_signals):
        return "HIGH"

    low_signals = ["what is", "where is", "define", "which file", "what does"]

    if (
        word_count <= 10
        and (any(signal in q for signal in low_signals)
             or q.startswith(("what ", "where ", "which ")))
    ):
        return "LOW"

    if word_count > 25 or " compare " in q or "impact" in q:
        return "HIGH"

    if word_count <= 15:
        return None  # use LLM fallback

    return "MEDIUM"


def classify_complexity(query: str, llm) -> tuple[str, float, str]:
    heuristic_result = classify_complexity_heuristic(query)

    if heuristic_result is not None:
        return heuristic_result, 1.0, "heuristic"

    prompt = COMPLEXITY_PROMPT.format(query=query)

    try:
        response = llm.invoke(
                        prompt,
                        max_tokens=256,
                        response_format={"type": "json_object"},
                    )
        data = json.loads(response.content)

        complexity = str(data.get("complexity", "MEDIUM")).upper()
        if complexity not in {"LOW", "MEDIUM", "HIGH"}:
            complexity = "MEDIUM"

        return (
            complexity,
            float(data.get("confidence", 0.5)),
            data.get("reason", "llm_fallback"),
        )
    except Exception as e:
        print(f"[complexity] LLM unavailable; using MEDIUM: {e}")
        return "MEDIUM", 0.0, "llm_fallback_unavailable"

#### adaptive cut off

In [19]:
DROPOFF_BY_COMPLEXITY = {"LOW": 0.08, "MEDIUM": 0.10, "HIGH": 0.12}

def adaptive_cutoff(reranked_results: list[dict], min_keep: int = 1,
                     max_keep: int = 10, complexity="MEDIUM") -> list[dict]:
    """
    Walk down reranked_score-sorted results and stop at the first sharp
    relative drop between consecutive scores. Assumes results are already
    sorted descending by 'rerank_score' (true for Reranker.rerank() output).
    """

    dropoff_ratio = DROPOFF_BY_COMPLEXITY.get(complexity, 0.12)
    min_keep = {"LOW": 3, "MEDIUM": 4, "HIGH": 5}.get(complexity, min_keep)
    if not reranked_results:
        return []
    if len(reranked_results) <= min_keep:
        return reranked_results

    kept = [reranked_results[0]]
    for i in range(1, min(len(reranked_results), max_keep)):
        if len(kept)>=min_keep:
            prev_score = reranked_results[i - 1]["rerank_score"]
            curr_score = reranked_results[i]["rerank_score"]
            # Cross-encoder scores can be negative/near-zero raw logits —
            # use absolute gap normalized by magnitude instead of a plain
            # ratio, which breaks when prev_score <= 0.
            denom = max(abs(prev_score), 1e-6)
            drop = (prev_score - curr_score) / denom
            if drop > dropoff_ratio and len(kept) >= min_keep:
                break
        kept.append(reranked_results[i])
    return kept

In [20]:
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

# Set budget conservatively (3500 tokens) to ensure prompt + response stay well under
# Groq's 8,000 TPM limit, even when running baseline & adaptive back-to-back.
MODEL_CONTEXT_WINDOW = 12000

def count_tokens(text: str) -> int:
    # Qwen tokenizer produces ~20% more tokens than cl100k_base for code.
    # Multiply by 1.25 to stay strictly conservative.
    return len(enc.encode(text))

def trim_to_token_budget(results, context_token_budget=None,
                          tpm_limit=MODEL_CONTEXT_WINDOW,
                          reserved_output_tokens=512,
                          prompt_overhead_tokens=150,
                          safety_margin=300):
    """Trim results so the full request fits within the Groq TPM limit.

    Budget = tpm_limit - reserved_output_tokens - prompt_overhead_tokens - safety_margin"""
    if context_token_budget is None:
        context_token_budget = tpm_limit - reserved_output_tokens - prompt_overhead_tokens - safety_margin
    context_token_budget = max(context_token_budget, 200)  # floor
    kept, total = [], 0
    for r in results:
        t = count_tokens(r['text'])
        if kept and total + t > context_token_budget:
            break
        kept.append(r)
        total += t
    return kept

In [21]:
RETRIEVAL_BUDGET = {
    "LOW": 10,
    "MEDIUM": 20,
    "HIGH": 30
}

In [22]:
user_query = "What is default ADMIN_EMAIL"
complexity, complexity_confidence, reason = classify_complexity(user_query, llm)

initial_k = RETRIEVAL_BUDGET.get(
    complexity,
    RETRIEVAL_BUDGET["MEDIUM"]
)

In [23]:
print("\n===== QUERY COMPLEXITY =====")
print("Complexity:", complexity)
print("Confidence:", complexity_confidence)
print("Reason:", reason)
print("Initial Retrieval K:", initial_k)


===== QUERY COMPLEXITY =====
Complexity: LOW
Confidence: 1.0
Reason: heuristic
Initial Retrieval K: 10


In [24]:
result = ragPipeline(
    user_query,
    hybrid_search=hybrid_search,
    reranker=re_ranker,
    llm=llm,
    top_k=initial_k
)

Retrieved documents: 25 documents (after filtering)


In [25]:
print(result)
print("Answer:", result['answer'])
print("Complexity:", result['complexity'], f"(confidence={result['complexity_confidence']})")
print("Reason:", result['complexity_reason'])
print("Retrieval K:", result['retrieval_k'], "-> Final chunks used:", result['final_chunk_count'])

{'answer': 'The default ADMIN_EMAIL is "shreerag99@gmail.com" (set in ADMIN_EMAIL = os.getenv("ADMIN_EMAIL", "shreerag99@gmail.com")).', 'sources': [{'source': 'temp/repos/Hybrid-Search-RAG/auth/firebase_auth.py', 'page': 'unknown', 'score': 1.0, 'preview': 'ADMIN_EMAIL = os.getenv("ADMIN_EMAIL", "shreerag99@gmail.com")...'}, {'source': 'temp/repos/Hybrid-Search-RAG/auth/firebase_auth.py', 'page': 'unknown', 'score': 0.9715725806451612, 'preview': 'def is_admin(email: str) -> bool:\n    """Check if the email matches the hardcoded admin email."""\n    return email.strip...'}, {'source': 'temp/repos/Hybrid-Search-RAG/api_server.py', 'page': 'unknown', 'score': 0.5809523809523809, 'preview': '\n    email = claims.get("email", "")\n    uid = claims.get("sub", "")\n    name = claims.get("name", "")\n    picture = cla...'}], 'confidence': 1.0, 'complexity': 'LOW', 'complexity_confidence': 1.0, 'complexity_reason': 'heuristic', 'retrieval_k': 10, 'final_chunk_count': 3}
Answer: The default AD

#### Cache layer

In [26]:
import hashlib
import json
import redis
import os

redis_client = redis.Redis(
    host=os.environ.get("REDIS_HOST", "localhost"),
    port=int(os.environ.get("REDIS_PORT", 6379)),
    password=os.environ.get("REDIS_PASSWORD"),
    decode_responses=True,
)

CACHE_TTL_SECONDS = 60 * 60 * 24  # 24h; tune based on how often the repo changes

def _cache_key(repo_name: str, commit_sha: str, query: str) -> str:
    raw = f"{repo_name}:{commit_sha}:{query.strip().lower()}"
    return "ragcache:" + hashlib.sha256(raw.encode()).hexdigest()

def ragPipeline_cached(query, repo_name, commit_sha, hybrid_search=None, reranker=None, llm=None,
                        top_k=None, top_n=None, min_score=0.2, return_context=False,
                        use_cache=True):
    key = _cache_key(repo_name, commit_sha, query)

    if use_cache:
        try:
            cached = redis_client.get(key)
            if cached:
                result = json.loads(cached)
                result["_cache_hit"] = True
                return result
        except redis.RedisError as e:
            print(f"[cache] Redis unavailable, skipping cache read: {e}")

    result = ragPipeline(query, hybrid_search=hybrid_search, reranker=reranker, llm=llm,
                          top_k=top_k, top_n=top_n, min_score=min_score,
                          return_context=return_context)
    result["_cache_hit"] = False

    if use_cache:
        try:
            redis_client.setex(key, CACHE_TTL_SECONDS, json.dumps(result))
        except redis.RedisError as e:
            print(f"[cache] Redis unavailable, skipping cache write: {e}")

    return result

#### Evaluation Harness

In [27]:
import csv
import time
import re

EVAL_SET = [
    # ---- LOW: single file / localized query ----
    {"query": "What language is this repository written in?", "expected_complexity": "LOW"},
    {"query": "Where is the FastAPI entry point defined?", "expected_complexity": "LOW"},

    # ---- MEDIUM: multi-function or single-subsystem reasoning ----
    {"query": "How does session creation and validation work in the FastAPI backend?", "expected_complexity": "MEDIUM"},
    {"query": "How does require_admin enforce role-based access control?", "expected_complexity": "MEDIUM"},

    # ---- HIGH: cross-file, cross-subsystem, architectural tracing ----
    {"query": "How does authentication and session management work across the frontend, API middleware, and database layers?", "expected_complexity": "HIGH"},
    {"query": "Trace a query from the Streamlit UI through the API to the final answer generation.", "expected_complexity": "HIGH"},
]

UNANSWERABLE_SET = [
    {"query": "What database engine is used to store user passwords?", "expected_complexity": "LOW", "unanswerable": True},
    {"query": "How does the project handle payment processing?", "expected_complexity": "LOW", "unanswerable": True},
]

EVAL_SET = EVAL_SET + UNANSWERABLE_SET

ABSTENTION_PHRASES = [
    "cannot be determined", "not defined in", "does not contain",
    "no information", "not covered", "cannot find", "not present in the context",
    "elsewhere in the codebase", "not included in the provided context",
    "not stored", "does not store", "no database", "not used for", "not implemented",
]

def is_abstention(answer: str) -> bool:
    a = answer.lower()
    return any(phrase in a for phrase in ABSTENTION_PHRASES)

def judge_answer(query: str, answer: str, context: str, llm) -> float:
    prompt = f"""Rate how well the ANSWER addresses the QUESTION using only the CONTEXT.
Return only one number: 0, 0.25, 0.5, 0.75, or 1.

QUESTION: {query}
CONTEXT: {context[:2000]}
ANSWER: {answer}

Score:"""

    try:
        response = llm.invoke(prompt)
        content = response.content

        if isinstance(content, list):
            content = "".join(
                part.get("text", "") if isinstance(part, dict) else str(part)
                for part in content
            )

        match = re.search(
                        r"\b(?:1(?:\.0+)?|0\.75|0\.5|0\.25|0)\b",
                        str(content),
                    )
        if not match:
            print(f"[judge] invalid response: {content!r}")
            return 0.0

        return float(match.group())

    except Exception as e:
        print(f"[judge] scoring failed: {e}")
        return 0.0

def score_unanswerable(query: str, answer: str, judge_llm=None) -> float:
    """Cheap keyword check first; LLM judge only for ambiguous cases
    (e.g. answers that correct the question's false premise instead
    of using hedge language)."""
    if is_abstention(answer):
        return 1.0  # clear keyword match — free, no LLM call needed

    if judge_llm is None:
        return 0.0  # no judge available — fall back to strict keyword-only

    prompt = f"""The QUESTION below assumes something that does not exist in this codebase.
A correct response either says the information isn't available, OR correctly explains
why the premise is false without inventing specifics not grounded in real context.
An INCORRECT response confidently states a specific, fabricated answer.

QUESTION: {query}
ANSWER: {answer}

Is this response correct (did it avoid fabricating an answer)? Reply only: yes or no."""

    try:
        response = judge_llm.invoke(prompt)
        return 1.0 if "yes" in response.content.lower() else 0.0
    except Exception as e:
        print(f"[abstention-judge] scoring failed, defaulting to 0.0: {e}")
        return 0.0

def run_eval(eval_set, hybrid_search, reranker, llm, judge_llm=None, sleep_between=1.5):
    if judge_llm is None:
        judge_llm = ChatGroq(model="openai/gpt-oss-20b", reasoning_format="hidden", reasoning_effort="low", temperature=0)
    rows = []

    for item in eval_set:
        query = item["query"]
        is_unanswerable = item.get("unanswerable", False)

        # Baseline: single ragPipeline call, fixed top_k/top_n, no adaptive cutoff/trim
        t0 = time.monotonic()
        baseline_out = ragPipeline(query, hybrid_search=hybrid_search, reranker=reranker,
                                    llm=llm, top_k=20, top_n=10, use_adaptive=False, return_context=True)
        baseline_latency = time.monotonic() - t0
        baseline_tokens = count_tokens(baseline_out.get("context", ""))

        time.sleep(sleep_between)  # give AstraDB/Groq room to breathe between calls

        # Adaptive: full pipeline as built
        t0 = time.monotonic()
        adaptive_out = ragPipeline(query, hybrid_search=hybrid_search, reranker=reranker,
                                    llm=llm, return_context=True, top_k=20,top_n=10)
        adaptive_latency = time.monotonic() - t0
        adaptive_tokens = count_tokens(adaptive_out.get("context", ""))

        if is_unanswerable:
            baseline_score = score_unanswerable(query, baseline_out["answer"], judge_llm)
            adaptive_score = score_unanswerable(query, adaptive_out["answer"], judge_llm)
        else:
            baseline_score = judge_answer(query, baseline_out["answer"], baseline_out.get("context", ""), judge_llm)
            adaptive_score = judge_answer(query, adaptive_out["answer"], adaptive_out.get("context", ""), judge_llm)

        if baseline_tokens <= 0:
            print(f"Skipping metrics for failed baseline: {query[:60]}")
            continue

        token_reduction_pct = round(
            100 * (1 - adaptive_tokens / baseline_tokens), 1
        )

        rows.append({
            "query": query,
            "unanswerable": is_unanswerable,
            "expected_complexity": item.get("expected_complexity"),
            "predicted_complexity": adaptive_out["complexity"],
            "baseline_tokens": baseline_tokens,
            "adaptive_tokens": adaptive_tokens,
            "token_reduction_pct": token_reduction_pct,
            "baseline_score": baseline_score,
            "adaptive_score": adaptive_score,
            "accuracy_retained_pct": (
    min(100.0, round(100 * adaptive_score / baseline_score, 1))
    if baseline_score > 0 else None
),
            "baseline_latency_s": round(baseline_latency, 2),
            "adaptive_latency_s": round(adaptive_latency, 2),
            "baseline_answer": baseline_out["answer"],
            "adaptive_answer": adaptive_out["answer"],
        })
        print(f"✓ {query[:60]}... | tokens {baseline_tokens}->{adaptive_tokens} | score {baseline_score:.2f}->{adaptive_score:.2f}")

        time.sleep(sleep_between)  # pace between full queries too

    # ... rest (CSV write + summary) unchanged

    if rows:
        with open("eval_results.csv", "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=rows[0].keys())
            writer.writeheader()
            writer.writerows(rows)

    answerable_rows = [r for r in rows if not r["unanswerable"]]
    unanswerable_rows = [r for r in rows if r["unanswerable"]]

    avg_token_reduction = (sum(r["token_reduction_pct"] for r in answerable_rows) / len(answerable_rows)) if answerable_rows else 0.0
    valid_accuracy_rows = [
        r for r in answerable_rows
        if r["accuracy_retained_pct"] is not None
    ]

    avg_accuracy_retained = (
        sum(r["accuracy_retained_pct"] for r in valid_accuracy_rows)
        / len(valid_accuracy_rows)
    ) if valid_accuracy_rows else 0.0
    abstention_rate = (
    sum(
        score_unanswerable(r["query"], r["adaptive_answer"], judge_llm)
        for r in unanswerable_rows
    ) / len(unanswerable_rows)
    if unanswerable_rows else None
)

    print(f"\n===== SUMMARY =====")
    print(f"Avg token reduction (answerable queries): {avg_token_reduction:.1f}%")
    print(f"Avg accuracy retained (answerable queries): {avg_accuracy_retained:.1f}%")
    if abstention_rate is not None:
        print(f"Correct abstention rate (unanswerable queries): {abstention_rate*100:.1f}%")
    
    if avg_accuracy_retained >= 95:
        print("Accept: token reduction improved while accuracy stayed acceptable.")
    else:
        print("Reject: increase min_keep and rerun.")
    return rows


In [28]:
import time
test_query = "Where is the FastAPI entry point defined?"

#baseline = ragPipeline(test_query, hybrid_search=hybrid_search, reranker=re_ranker,
#                        llm=llm, top_k=20, top_n=10, use_adaptive=False, return_context=True)  # ← top_n=5 to top_n=10

#adaptive = ragPipeline(test_query, hybrid_search=hybrid_search, reranker=re_ranker,
#                        llm=llm, return_context=True)

#baseline_tokens = count_tokens(baseline.get("context", ""))
#adaptive_tokens = count_tokens(adaptive.get("context", ""))

#print("=== BASELINE (use_adaptive=False) ===")
#print("Answer:", baseline["answer"])
#print("Tokens:", baseline_tokens)
#print("Chunks:", baseline["final_chunk_count"])

#print("\n=== ADAPTIVE ===")
#print("Answer:", adaptive["answer"])
#print("Complexity:", adaptive["complexity"])
#print("Tokens:", adaptive_tokens)
#print("Chunks:", adaptive["final_chunk_count"])

#print("\n=== DIFF CHECK ===")
#print("Token reduction:", round(100 * (1 - adaptive_tokens / max(baseline_tokens, 1)), 1), "%")
#print("Are baseline and adaptive identical?", baseline_tokens == adaptive_tokens and baseline["final_chunk_count"] == adaptive["final_chunk_count"])

In [29]:
#import requests, os
#
#resp = requests.get(
#    "https://api.groq.com/openai/v1/models",
#    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"}
#)
#for m in resp.json()["data"]:
#    if m["id"] == "allam-2-7b":
#        print(m)

In [30]:
#import requests, os

#resp = requests.get(
#    "https://api.groq.com/openai/v1/models",
#    headers={"Authorization": f"Bearer {os.environ['GROQ_API_KEY']}"}
#)
#for m in resp.json()["data"]:
#    print(m["id"])

In [31]:
#judge_llm = ChatGroq(model="openai/gpt-oss-20b", reasoning_format="hidden", reasoning_effort="low", temperature=0)
judge_llm = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0,
    max_tokens=128,
    reasoning_effort="minimal",
    api_key=os.getenv("OPENAI_API_KEY"),
)
test_score = judge_answer(
    "What language is this repository written in?",
    "Python.",
    "This is a Python codebase using FastAPI and Streamlit.",
    judge_llm
)
print(test_score)  # should print something like 1.0, not 0.0 with a [judge] error above it

1.0


In [32]:
print(classify_complexity(
    "How does session creation and validation work in the FastAPI backend?",
    classifier_llm,
))

('MEDIUM', 0.46, 'Understanding session creation and validation in a FastAPI backend typically involves multiple files/components: authentication/session management module, middleware or dependencies, user models, and possibly database interactions or token handling. It often requires tracing the data flow from login/refresh/token issuance through to validation on protected routes, which crosses several files and may involve architectural choices (e.g., JWT vs server-side sessions). Not guaranteed to need the entire repo, but usually more than a single file.')


In [33]:
print(judge_answer(
    "What language is this repository written in?",
    "Python.",
    "This is a Python codebase using FastAPI and Streamlit.",
    judge_llm,
))

1.0


In [34]:
#judge_llm = ChatGroq(model="openai/gpt-oss-20b", reasoning_format="hidden", reasoning_effort="low", temperature=0)
judge_llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    max_tokens=16,
    api_key=os.getenv("OPENAI_API_KEY"),
)
results = run_eval(EVAL_SET, hybrid_search=hybrid_search, reranker=re_ranker, llm=llm, judge_llm=judge_llm)

Retrieved documents: 25 documents (after filtering)
Retrieved documents: 25 documents (after filtering)
✓ What language is this repository written in?... | tokens 5507->5507 | score 1.00->0.75
Retrieved documents: 25 documents (after filtering)
Retrieved documents: 25 documents (after filtering)
✓ Where is the FastAPI entry point defined?... | tokens 4029->1193 | score 1.00->1.00
Retrieved documents: 25 documents (after filtering)
Retrieved documents: 25 documents (after filtering)
✓ How does session creation and validation work in the FastAPI... | tokens 3227->1277 | score 1.00->1.00
Retrieved documents: 25 documents (after filtering)
Retrieved documents: 25 documents (after filtering)
✓ How does require_admin enforce role-based access control?... | tokens 2406->2406 | score 1.00->1.00
Retrieved documents: 25 documents (after filtering)
Retrieved documents: 25 documents (after filtering)
✓ How does authentication and session management work across t... | tokens 3396->3328 | score 1.00

In [35]:
print("ANSWER QUALITY CHECK")
print("=" * 80)

for row in results:
    if row["unanswerable"]:
        continue

    changed = (
        row["baseline_answer"].strip()
        != row["adaptive_answer"].strip()
    )

    print("\nQuestion:")
    print(row["query"])

    print("\nBaseline answer:")
    print(row["baseline_answer"])

    print("\nAdaptive answer:")
    print(row["adaptive_answer"])

    print("\nToken count:")
    print(
        f"{row['baseline_tokens']} -> "
        f"{row['adaptive_tokens']} "
        f"({row['token_reduction_pct']}% reduction)"
    )

    print("\nAnswer changed:", changed)
    print(
        "Scores:",
        f"{row['baseline_score']:.2f} -> "
        f"{row['adaptive_score']:.2f}"
    )

    print("-" * 80)

ANSWER QUALITY CHECK

Question:
What language is this repository written in?

Baseline answer:
This repository is written in Python — evidenced by .py files and Python constructs like import statements, classes, type hints (e.g., "from typing import List, Dict, Any", FastAPI app, and functions/classes such as RAGRetreiever, lifespan, and main()).

Adaptive answer:
The repository is written in Python (files use .py modules, FastAPI, asynccontextmanager, import statements like "import os", and Python classes/functions such as RAGRetreiver, lifespan, and main()).

Token count:
5507 -> 5507 (0.0% reduction)

Answer changed: True
Scores: 1.00 -> 0.75
--------------------------------------------------------------------------------

Question:
Where is the FastAPI entry point defined?

Baseline answer:
The FastAPI application entry point is defined in api_server.py where `app = FastAPI(...)` is created (and the server is started with `uvicorn api_server:app --host 0.0.0.0 --port 8000`).

Adapt

In [36]:
for r in results:
    if r["unanswerable"]:
        print(f"Query: {r['query']}")
        print(f"Baseline answer: {r['baseline_answer']}")
        print(f"Adaptive answer: {r['adaptive_answer']}")
        print()

Query: What database engine is used to store user passwords?
Baseline answer: User authentication uses Firebase (Firebase Auth) for sign-in and token verification (see verify_firebase_token in the Firebase Authentication module and /auth endpoints), so passwords are managed by Firebase Auth rather than stored in this app.
Adaptive answer: The repo stores server-side sessions in an in-memory dict (_sessions) and does not show any persistent user password database; there is no password-storage engine referenced in the provided context.

Query: How does the project handle payment processing?
Baseline answer: Cannot be determined from the provided repository context.
Adaptive answer: Cannot be determined from the provided repository context.



In [37]:
baseline_score = score_unanswerable(
    "What database engine is used to store user passwords?",
    "User passwords are not stored in this app; authentication uses Firebase Authentication (verified via verify_firebase_token in the Firebase auth module) and server-side sessions are kept in an in-memory _sessions dict (see create_session / get_session).",
    judge_llm
)
adaptive_score = score_unanswerable(
    "What database engine is used to store user passwords?",
    "User authentication uses Firebase (Firebase Auth) for sign-in and token verification via verify_firebase_token (google.oauth2.id_token), so passwords are not stored in this app's DB.",
    judge_llm
)
print("Baseline:", baseline_score, "| Adaptive:", adaptive_score)

Baseline: 1.0 | Adaptive: 1.0


In [38]:
judge_llm = ChatOpenAI(
    model="gpt-4.1-mini",
    temperature=0,
    max_tokens=16,
    api_key=os.getenv("OPENAI_API_KEY"),
)
print(judge_answer(
    "What language is this repository written in?",
    "Python.",
    "This is a Python codebase using FastAPI and Streamlit.",
    judge_llm,
))

1.0


In [39]:
for query in [
    "Where is the FastAPI entry point defined?",
    "How does session creation and validation work in the FastAPI backend?",
    "Trace a query from the Streamlit UI through the API to the final answer generation.",
]:
    print(query)
    print(ragPipeline(
        query,
        hybrid_search=hybrid_search,
        reranker=re_ranker,
        llm=llm,
        return_context=True,
    ))

Where is the FastAPI entry point defined?
Retrieved documents: 25 documents (after filtering)
{'answer': 'The FastAPI entry point is defined in api_server.py where the FastAPI app is created with `app = FastAPI(...)` (near the bottom of the shown file).', 'sources': [{'source': 'temp/repos/Hybrid-Search-RAG/README.md', 'page': 'unknown', 'score': 0.9462686567164179, 'preview': '## 📁 Project Structure\n\n```\nhybrid-search-RAG-Application/\n├── api_server.py              # FastAPI REST backend server\n...'}, {'source': 'temp/repos/Hybrid-Search-RAG/api_server.py', 'page': 'unknown', 'score': 0.9116483516483517, 'preview': '"""\nFastAPI backend for the Hybrid Search RAG application.\n\nEndpoints:\n  PUBLIC (no auth):\n    POST /api/query        — ...'}, {'source': 'temp/repos/Hybrid-Search-RAG/api_server.py', 'page': 'unknown', 'score': 0.9657071960297765, 'preview': 'app = FastAPI(\n    title="Hybrid Search RAG API",\n    description="FastAPI backend for the Hybrid Search RAG Applicatio

In [40]:
result = ragPipeline(
    "Where is the FastAPI entry point defined?",
    hybrid_search=hybrid_search,
    reranker=re_ranker,
    llm=llm,
    use_adaptive=False,
    return_context=True,
)

print(result["answer"])

Retrieved documents: 25 documents (after filtering)
The FastAPI entry point is defined in api_server.py where the app instance is created: app = FastAPI(..., lifespan=lifespan) and the module is run with uvicorn via the command shown: uvicorn api_server:app --host 0.0.0.0 --port 8000.


In [41]:
test_prompt = """The context says:
api_server.py contains the FastAPI backend.
The file contains app = FastAPI(...).

Question: Where is the FastAPI entry point defined?

Answer in one sentence."""
print(llm.invoke(test_prompt).content)

The FastAPI entry point is defined in api_server.py where the application instance is created with app = FastAPI(...).


In [42]:
adaptive_result = ragPipeline(
    "Where is the FastAPI entry point defined?",
    hybrid_search=hybrid_search,
    reranker=re_ranker,
    llm=llm,
    use_adaptive=True,
    return_context=True,
)

print(adaptive_result["answer"])
print("Complexity:", adaptive_result["complexity"])
print("Retrieval k:", adaptive_result["retrieval_k"])
print("Final chunks:", adaptive_result["final_chunk_count"])

Retrieved documents: 25 documents (after filtering)
The FastAPI entry point is defined in api_server.py where `app = FastAPI(...)` is created (near the bottom of the shown file).
Complexity: LOW
Retrieval k: 10
Final chunks: 3


In [43]:
baseline_result = ragPipeline(
    "Where is the FastAPI entry point defined?",
    hybrid_search=hybrid_search,
    reranker=re_ranker,
    llm=llm,
    use_adaptive=False,
    return_context=True,
)

baseline_tokens = count_tokens(
    baseline_result.get("context", "")
)

adaptive_tokens = count_tokens(
    adaptive_result.get("context", "")
)

print("BASELINE:")
print(baseline_result["answer"])
print("Tokens:", baseline_tokens)

print("\nADAPTIVE:")
print(adaptive_result["answer"])
print("Tokens:", adaptive_tokens)

if baseline_tokens:
    reduction = (
        1 - adaptive_tokens / baseline_tokens
    ) * 100
    print("\nToken reduction:", round(reduction, 2), "%")

Retrieved documents: 25 documents (after filtering)
BASELINE:
The FastAPI app entry point is defined in api_server.py as the `app = FastAPI(...)` instance (created near the bottom of api_server.py) and launched with `uvicorn api_server:app --host 0.0.0.0 --port 8000` (shown in the file).
Tokens: 4064

ADAPTIVE:
The FastAPI entry point is defined in api_server.py where `app = FastAPI(...)` is created (near the bottom of the shown file).
Tokens: 1193

Token reduction: 70.64 %


In [44]:
print("FastAPI evidence retained:",
      "app = FastAPI" in adaptive_result["context"])

FastAPI evidence retained: True
